In [7]:
report = "Q4 results: Revenue +12% QoQ, churn rose from 4.1% → 5.3%, NPS fell from 41 → 33. Biggest driver: support response time increased from 6h → 18h after vendor switch. Enterprise deals grew (3 new logos), but mid-market downgrades increased. Engineering missed 2 delivery dates due to onboarding 4 new hires. Cash runway: 14 months. Recommendation from finance: slow hiring and renegotiate support vendor contract."

In [10]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic
load_dotenv()

client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
response = client.messages.create(
    model="claude-opus-4-20250514",
    max_tokens=4000,
    temperature=0.2,
    messages=[
        {"role": "user", "content": prompt_without_tags}
    ]
)
try:
    parsed = json.loads(result)
    print(json.dumps(parsed, indent=2))
except:
# Otherwise just print normally
    print(result)


NameError: name 'result' is not defined

In [13]:
response.content[0].text

"## Q4 Summary\n\n**Performance:** Revenue grew 12% QoQ driven by 3 new enterprise logos, but warning signs emerged with churn jumping to 5.3% and NPS dropping 8 points to 33.\n\n**Root Cause:** Support vendor switch tripled response times (6h → 18h), driving customer dissatisfaction and mid-market downgrades.\n\n**Additional Challenges:** Engineering missed 2 delivery dates while onboarding 4 new hires. Cash runway stands at 14 months.\n\n---\n\n**Email to CEO:**\n\nSubject: Q4 Results - Strong Revenue Growth but Customer Health Declining\n\nHi [CEO],\n\nQ4 delivered 12% revenue growth with 3 new enterprise wins, but we're facing significant customer retention challenges that need immediate attention.\n\n**Key Risks:**\n- Churn increased 29% (4.1% → 5.3%) with NPS dropping to 33\n- Support response times tripled after vendor switch - primary driver of dissatisfaction\n- Mid-market segment showing increased downgrades\n- Engineering delays impacting product roadmap\n- 14-month runway r

# Exercise 01: XML Structure

## The Scenario
You're building a feature that analyzes customer support tickets and:
1. Categorizes them (bug, feature request, billing, general inquiry)
2. Extracts key entities (product mentioned, urgency level, customer tier)
3. Suggests a response template

**Goal**: Compare naive vs optimized prompting with XML structure

In [ ]:
# Sample support ticket
ticket = """Subject: URGENT - App crashes when I try to export

Hi,

I've been a Pro subscriber for 2 years and I'm extremely frustrated.
Every time I click the export button in the dashboard, the app freezes
and then crashes. I've tried Chrome and Firefox. This is blocking my
quarterly report which is due tomorrow!

Please help ASAP.

- Sarah
"""

## Part 1: Naive Prompt (No XML Structure)
This is what most people write first - mixing context and instructions together

In [ ]:
import json

# Naive prompt - unstructured
naive_prompt = f"""
Analyze this support ticket and categorize it. Extract the product, urgency level, and customer tier. 
Also suggest a response template. Return the results as JSON.

{ticket}
"""

print("NAIVE PROMPT:")
print(naive_prompt)
print("\n" + "="*80 + "\n")

In [ ]:
# Test naive prompt
response_naive = client.messages.create(
    model="claude-opus-4-20250514",
    max_tokens=2000,
    temperature=0.2,
    messages=[
        {"role": "user", "content": naive_prompt}
    ]
)

result_naive = response_naive.content[0].text
print("NAIVE RESPONSE:")
print(result_naive)
print("\n" + "="*80 + "\n")

# Try to parse as JSON
try:
    parsed_naive = json.loads(result_naive)
    print("✅ JSON is valid")
    print(json.dumps(parsed_naive, indent=2))
except json.JSONDecodeError as e:
    print("❌ JSON parsing failed!")
    print(f"Error: {e}")
    print("\nLikely issues: markdown wrapper, explanatory text, or invalid JSON")

## Part 2: Optimized Prompt (With XML Structure)
Now we separate context, instructions, constraints, and output format using XML tags

In [ ]:
# Optimized prompt - with XML structure
optimized_prompt = f"""<context>
{ticket}
</context>

<instructions>
1. Categorize the ticket into one of the allowed categories
2. Extract key entities from the text
3. Generate an appropriate response template
</instructions>

<constraints>
- Categories must be one of: bug, feature, billing, general
- Urgency must be inferred from language and stated deadlines
- If an entity cannot be determined, use null
</constraints>

<output_format>
Return ONLY valid JSON with this structure:
{{
  "category": "bug" | "feature" | "billing" | "general",
  "entities": {{
    "product": string | null,
    "urgency": "low" | "medium" | "high" | "critical",
    "customer_tier": string | null
  }},
  "suggested_template": string
}}
No markdown code blocks. No explanatory text. Just the JSON object.
</output_format>
"""

print("OPTIMIZED PROMPT:")
print(optimized_prompt)
print("\n" + "="*80 + "\n")

In [ ]:
# Test optimized prompt
response_optimized = client.messages.create(
    model="claude-opus-4-20250514",
    max_tokens=2000,
    temperature=0.2,
    messages=[
        {"role": "user", "content": optimized_prompt}
    ]
)

result_optimized = response_optimized.content[0].text
print("OPTIMIZED RESPONSE:")
print(result_optimized)
print("\n" + "="*80 + "\n")

# Try to parse as JSON
try:
    parsed_optimized = json.loads(result_optimized)
    print("✅ JSON is valid")
    print(json.dumps(parsed_optimized, indent=2))
except json.JSONDecodeError as e:
    print("❌ JSON parsing failed!")
    print(f"Error: {e}")

## Comparison: What to Observe

Run both cells multiple times and notice:

**Naive Prompt Issues:**
- 🔴 Often wraps JSON in markdown code blocks (```json ... ```)
- 🔴 May add explanatory text before/after JSON
- 🔴 Inconsistent entity extraction across runs
- 🔴 Harder to parse programmatically

**Optimized Prompt Benefits:**
- ✅ Returns pure JSON (no wrappers)
- ✅ Consistent structure every time
- ✅ Follows exact schema specified
- ✅ No extra explanatory text
- ✅ Ready for `json.loads()` without cleanup

**Key Takeaway:** XML tags help Claude understand the STRUCTURE of your request, leading to more reliable and consistent outputs.

# Exercise 02: Role Separation (System vs User)

## The Scenario
You're building a code review assistant with a senior engineer persona that:
1. Reviews Python code for bugs and style issues
2. Maintains a pragmatic perspective (not pedantic)
3. Explains the "why" behind suggestions

**Goal**: Compare putting role in user message vs system prompt

In [ ]:
# Sample code to review
code_to_review = """def process_users(users):
    result = []
    for i in range(len(users)):
        user = users[i]
        if user['status'] == 'active':
            if user['age'] >= 18:
                if user['email'] != None:
                    result.append({
                        'name': user['name'],
                        'email': user['email'],
                        'type': 'adult'
                    })
    return result
"""

## Part 1: Naive Approach (Role in User Message)

In [ ]:
# Naive approach - everything in user message
naive_prompt = f"""You are a senior staff engineer with 15 years of Python experience. 
You're pragmatic and focus on real issues, not nitpicks. You explain the "why" behind suggestions.
You keep feedback actionable and maintain a professional but direct tone.

Review this code and identify bugs, style issues, and suggest improvements:

{code_to_review}
"""

response_naive = client.messages.create(
    model="claude-opus-4-20250514",
    max_tokens=2000,
    temperature=0.2,
    messages=[
        {"role": "user", "content": naive_prompt}
    ]
)

print("NAIVE APPROACH (role in user message):")
print(response_naive.content[0].text)
print("\n" + "="*80 + "\n")

## Part 2: Optimized Approach (System vs User Separation)

In [ ]:
# Optimized approach - system vs user separation
system_prompt = """You are a senior staff engineer with deep Python expertise.
You're pragmatic—you focus on issues that matter, not style nitpicks.
You explain the "why" behind suggestions and keep feedback actionable.
Your tone is direct and professional."""

user_message = f"""<code>
{code_to_review}
</code>

<task>
Review this code. Identify bugs, significant style issues, and suggest improvements.
Focus on the 3 most impactful issues.
</task>
"""

response_optimized = client.messages.create(
    model="claude-opus-4-20250514",
    max_tokens=2000,
    temperature=0.2,
    system=system_prompt,  # ← Key difference: role goes in system
    messages=[
        {"role": "user", "content": user_message}
    ]
)

print("OPTIMIZED APPROACH (role in system prompt):")
print(response_optimized.content[0].text)
print("\n" + "="*80 + "\n")

## Part 3: Bonus - No System Prompt (Generic Assistant)

In [ ]:
# Bonus - Test with NO system prompt
response_no_system = client.messages.create(
    model="claude-opus-4-20250514",
    max_tokens=2000,
    temperature=0.2,
    messages=[
        {"role": "user", "content": f"Review this code:\n\n{code_to_review}"}
    ]
)

print("NO SYSTEM PROMPT (generic assistant):")
print(response_no_system.content[0].text)
print("\n" + "="*80 + "\n")

## Comparison: What to Observe

Run all three cells and compare:

**Naive (Role in User Message):**
- 🔴 Claude may "acknowledge" the role ("As a senior engineer...")
- 🔴 Wastes tokens explaining it's being pragmatic
- 🔴 Less consistent persona across runs

**Optimized (System Prompt):**
- ✅ Claude **IS** the senior engineer - no role-play
- ✅ Direct feedback, no meta-commentary
- ✅ Consistent expert tone
- ✅ Cleaner, more focused responses

**No System Prompt:**
- 🔴 Generic helpful assistant tone
- 🔴 Less opinionated, more cautious
- 🔴 May miss nuanced issues a senior engineer would catch

**Key Takeaway:** 
- **System prompt** = persistent identity/persona
- **User message** = task-specific instructions
- Separating these makes Claude embody the role rather than perform it

# Exercise 03: Explicit Intent (Why vs What)

## The Scenario
You're asking Claude to write tests for a discount calculation function. This function is critical to billing.

**Goal**: Compare "just ask for tests" vs "explain WHY tests matter"

In [ ]:
# Function to test
function_code = '''def calculate_discount(price: float, user_tier: str, promo_code: str | None = None) -> float:
    """
    Calculate final price after applying discounts.
    
    Tiers: 'basic' (0%), 'silver' (5%), 'gold' (10%), 'platinum' (15%)
    Promo codes: 'SAVE10' (10% off), 'SAVE20' (20% off)
    Discounts stack multiplicatively.
    """
    tier_discounts = {'basic': 0, 'silver': 0.05, 'gold': 0.10, 'platinum': 0.15}
    promo_discounts = {'SAVE10': 0.10, 'SAVE20': 0.20}
    
    if user_tier not in tier_discounts:
        raise ValueError(f"Invalid tier: {user_tier}")
    
    discount = tier_discounts[user_tier]
    if promo_code and promo_code in promo_discounts:
        # Stack multiplicatively
        discount = 1 - (1 - discount) * (1 - promo_discounts[promo_code])
    
    return round(price * (1 - discount), 2)
'''

print("Function to test:")
print(function_code)

## Part 1: Naive Approach (Just ask for tests)

In [ ]:
# Naive prompt - just ask for tests
naive_prompt = f"""Write unit tests for this function:

{function_code}
"""

response_naive = client.messages.create(
    model="claude-opus-4-20250514",
    max_tokens=3000,
    temperature=0.2,
    messages=[
        {"role": "user", "content": naive_prompt}
    ]
)

print("NAIVE APPROACH (no context on WHY tests matter):")
print(response_naive.content[0].text)
print("\n" + "="*80 + "\n")

## Part 2: Optimized Approach (Explain WHY tests matter)

In [ ]:
# Optimized prompt - explain the WHY
optimized_prompt = f"""<function>
{function_code}
</function>

<task>
Write comprehensive unit tests for this discount calculation function.
</task>

<context>
This function is critical to our billing system:
- Bugs in discount calculation = revenue loss or customer disputes
- Tests are the single source of truth for expected behavior
- The multiplicative stacking is often misunderstood by new developers
</context>

<requirements>
Tests must cover:
1. All user tiers (basic, silver, gold, platinum)
2. Promo code stacking behavior (multiplicative, NOT additive)
3. Invalid inputs (bad tier, invalid promo code)
4. Edge cases (zero price, None promo code)
5. Verify exact decimal values (rounding matters for money)
</requirements>

<unacceptable>
- Tests that only check happy path
- Missing coverage for multiplicative discount stacking
- Tests that would pass even if the function was buggy
- Generic "it runs" tests without value assertions
</unacceptable>
"""

response_optimized = client.messages.create(
    model="claude-opus-4-20250514",
    max_tokens=3000,
    temperature=0.2,
    messages=[
        {"role": "user", "content": optimized_prompt}
    ]
)

print("OPTIMIZED APPROACH (with WHY and context):")
print(response_optimized.content[0].text)
print("\n" + "="*80 + "\n")

## Part 3: Analyze Test Coverage

In [ ]:
# Analyze test coverage
def analyze_tests(response_text):
    """Quick analysis of test coverage"""
    tests = {
        'All tiers tested': 'silver' in response_text and 'gold' in response_text and 'platinum' in response_text,
        'Promo code stacking': 'SAVE10' in response_text or 'SAVE20' in response_text,
        'Error handling': 'ValueError' in response_text or 'invalid' in response_text.lower(),
        'Edge cases': 'zero' in response_text.lower() or 'None' in response_text or '0.0' in response_text,
        'Multiplicative calc': 'multiplicative' in response_text.lower() or '0.765' in response_text or '76.5' in response_text
    }
    return tests

print("NAIVE TEST COVERAGE:")
naive_analysis = analyze_tests(response_naive.content[0].text)
for test, covered in naive_analysis.items():
    print(f"  {test}: {'✅' if covered else '❌'}")

print("\nOPTIMIZED TEST COVERAGE:")
optimized_analysis = analyze_tests(response_optimized.content[0].text)
for test, covered in optimized_analysis.items():
    print(f"  {test}: {'✅' if covered else '❌'}")

# Count total coverage
naive_score = sum(naive_analysis.values())
optimized_score = sum(optimized_analysis.values())

print(f"\nNaive Coverage: {naive_score}/5 ({naive_score/5*100:.0f}%)")
print(f"Optimized Coverage: {optimized_score}/5 ({optimized_score/5*100:.0f}%)")

## Comparison: What to Observe

**Naive Approach (What only):**
- 🔴 May write 3-5 basic tests
- 🔴 Often misses edge cases
- 🔴 Might not test multiplicative stacking explicitly
- 🔴 "Happy path" focused

**Optimized Approach (What + Why):**
- ✅ More comprehensive test suite (8-12 tests)
- ✅ Explicitly tests multiplicative vs additive stacking
- ✅ Covers all edge cases mentioned
- ✅ Tests are defensive - catch real bugs
- ✅ Better test names that reflect purpose

**Key Takeaway:** 
When you explain **WHY** something matters, Claude understands:
- **Intent** - prevent revenue loss, not just "write tests"
- **Priorities** - multiplicative stacking is critical
- **Standards** - what "good enough" means

This leads to more thoughtful, comprehensive outputs.

## Part 4: BETTER Example - Where "Why" Actually Matters

Let's try an **ambiguous** task where the "why" makes a dramatic difference

In [ ]:
# Ambiguous scenario: user feedback analysis
feedback_data = """
User feedback from last 30 days:

"The app is slow" - 45 mentions
"Love the new export feature!" - 23 mentions  
"Crashes on large files" - 12 mentions
"UI is confusing" - 34 mentions
"Customer support was amazing" - 8 mentions
"Pricing is too high" - 67 mentions
"Can't find the delete button" - 15 mentions
"Mobile version is broken" - 28 mentions
"""

print("TASK: Analyze this feedback and create an action plan")
print(feedback_data)

In [ ]:
# Naive: No context about priorities
naive_feedback_prompt = f"""Analyze this user feedback and create an action plan:

{feedback_data}
"""

response_naive_feedback = client.messages.create(
    model="claude-opus-4-20250514",
    max_tokens=2000,
    temperature=0.2,
    messages=[
        {"role": "user", "content": naive_feedback_prompt}
    ]
)

print("NAIVE (no context):")
print(response_naive_feedback.content[0].text)
print("\n" + "="*80 + "\n")

In [ ]:
# Scenario A: Growth-focused startup
growth_context = f"""<feedback>
{feedback_data}
</feedback>

<task>
Analyze this feedback and create a prioritized action plan.
</task>

<context>
We're a growth-stage startup preparing for Series B fundraising in 60 days.
- Our board cares about: user acquisition, expansion revenue, and retention metrics
- Engineering team: 2 developers (both working on new features for enterprise clients)
- Support team: 1 person, already overwhelmed
- Burn rate: $150k/month, 8 months runway
- Key metric: need to show 15% month-over-month user growth

WHY THIS MATTERS:
We need to prioritize issues that directly impact our fundraising story.
Issues that hurt growth or signal to investors that we're not enterprise-ready are critical.
We can't fix everything - need to focus on what moves the needle for fundraising.
</context>

<constraints>
- Can only tackle 2-3 issues this quarter
- Engineering time is the bottleneck
- Must show progress in 60 days for investor meetings
</constraints>
"""

response_growth = client.messages.create(
    model="claude-opus-4-20250514",
    max_tokens=2000,
    temperature=0.2,
    messages=[
        {"role": "user", "content": growth_context}
    ]
)

print("WITH CONTEXT (Growth-stage startup, fundraising focus):")
print(response_growth.content[0].text)
print("\n" + "="*80 + "\n")

In [ ]:
# Scenario B: Profitability-focused bootstrap
profitability_context = f"""<feedback>
{feedback_data}
</feedback>

<task>
Analyze this feedback and create a prioritized action plan.
</task>

<context>
We're a bootstrapped SaaS with 5,000 paying customers, profitable for 2 years.
- Philosophy: sustainable growth, low churn, happy customers > explosive growth
- Team: founder + 1 part-time developer + outsourced support
- Profit margin: 60%, reinvesting 20% into product
- Churn rate: 2% monthly (industry average is 5%)
- Core customers: small businesses who value stability over features

WHY THIS MATTERS:
Our competitive advantage is reliability and customer happiness.
We'd rather fix what's broken than add new features.
Churn is our biggest risk - if we lose customers, we lose our profit engine.
Pricing complaints are noise unless they're causing cancellations.
</context>

<constraints>
- Limited dev time (15 hours/week)
- Can't hire more people without hurting profitability
- Customer support quality is our differentiator
</constraints>
"""

response_profitability = client.messages.create(
    model="claude-opus-4-20250514",
    max_tokens=2000,
    temperature=0.2,
    messages=[
        {"role": "user", "content": profitability_context}
    ]
)

print("WITH CONTEXT (Bootstrapped, profitability + retention focus):")
print(response_profitability.content[0].text)
print("\n" + "="*80 + "\n")

## THIS is Where "Why" Makes a HUGE Difference

**Same Data. Same Task. Completely Different Recommendations.**

### What You'll See:

**Naive Response:**
- Generic prioritization (probably by volume: pricing > performance > UI)
- Balanced across all issues
- No strategic thinking

**Growth-Focused Context:**
- Will prioritize: crashes, mobile issues (block enterprise deals)
- Will IGNORE: pricing complaints (can't change pricing before fundraising)
- Strategic: focus on "enterprise-ready" signals

**Profitability-Focused Context:**
- Will prioritize: crashes, UI confusion (causes churn)
- Will IGNORE: pricing complaints (unless causing cancellations)
- Strategic: protect existing happy customers

**The Difference:** Context changes **which problems matter** and **what success looks like**.

# Exercise 04: Constraint Priority

## The Scenario
You need to refactor code with **competing constraints**. When constraints conflict, what does Claude prioritize?

**Goal**: Compare flat constraints vs explicit priority ordering

In [ ]:
# Code to refactor
session_code = '''import time

class SessionManager:
    def __init__(self):
        self.sessions = {}  # user_id -> session_data
        self.MAX_SESSIONS = 1000

    def create_session(self, user_id, data):
        if len(self.sessions) >= self.MAX_SESSIONS:
            # Remove oldest session
            oldest = min(self.sessions.items(), key=lambda x: x[1]['created_at'])
            del self.sessions[oldest[0]]
        self.sessions[user_id] = {
            'data': data,
            'created_at': time.time(),
            'last_access': time.time()
        }
        return True

    def get_session(self, user_id):
        if user_id in self.sessions:
            self.sessions[user_id]['last_access'] = time.time()
            return self.sessions[user_id]['data']
        return None

    def cleanup_expired(self, max_age=3600):
        now = time.time()
        expired = [uid for uid, s in self.sessions.items()
                   if now - s['last_access'] > max_age]
        for uid in expired:
            del self.sessions[uid]
'''

print("Code to refactor:")
print(session_code)

## Part 1: Naive Approach (Flat Constraints with Strong Language)

In [ ]:
# Naive: All constraints with strong language, no priority
naive_refactor_prompt = f"""Refactor this code:

{session_code}

CONSTRAINTS:
- You MUST NOT introduce any security vulnerabilities
- You MUST maintain backward compatibility - existing code using this class MUST still work
- You MUST keep the diff as small as possible
- You MUST improve performance wherever possible
- You MUST make the code more readable and clear

Follow ALL these constraints.
"""

response_naive_refactor = client.messages.create(
    model="claude-opus-4-20250514",
    max_tokens=2500,
    temperature=0.2,
    messages=[
        {"role": "user", "content": naive_refactor_prompt}
    ]
)

print("NAIVE (flat constraints, no priority):")
print(response_naive_refactor.content[0].text)
print("\n" + "="*80 + "\n")

## Part 2: Optimized Approach (Explicit Priority Ordering)

In [ ]:
# Optimized: Same constraints, but with priority ordering
optimized_refactor_prompt = f"""<code>
{session_code}
</code>

<task>
Refactor this code to improve it.
</task>

<constraints>
When these constraints conflict, follow the lower priority number:

Priority 1 (CRITICAL): Do not introduce security vulnerabilities
Priority 2 (HIGH): Maintain backward compatibility - existing code calling these methods must still work unchanged
Priority 3 (MEDIUM): Keep the diff as small as possible - don't change things unnecessarily
Priority 4 (LOW): Improve performance where possible without violating priorities 1-3
Priority 5 (NICE-TO-HAVE): Improve readability if it doesn't conflict with priorities 1-4

If you must violate a lower-priority constraint to satisfy a higher one, briefly explain the tradeoff.
</constraints>

<output>
Provide the refactored code and explain any tradeoffs you made.
</output>
"""

response_optimized_refactor = client.messages.create(
    model="claude-opus-4-20250514",
    max_tokens=2500,
    temperature=0.2,
    messages=[
        {"role": "user", "content": optimized_refactor_prompt}
    ]
)

print("OPTIMIZED (explicit priority ordering):")
print(response_optimized_refactor.content[0].text)
print("\n" + "="*80 + "\n")

## Comparison: What to Observe

Run both cells multiple times and look for:

**Naive Approach (flat constraints):**
- 🔴 Might rename methods "for clarity" → breaks backward compatibility
- 🔴 Could add parameters to methods → breaks existing code
- 🔴 May do large rewrites → violates minimal diff
- 🔴 Inconsistent decisions across runs
- 🔴 No explanation of tradeoffs

**Optimized Approach (priority ordering):**
- ✅ Respects backward compatibility strictly (Priority 2)
- ✅ Makes targeted, minimal changes (Priority 3)
- ✅ Only improves performance if it doesn't break above (Priority 4)
- ✅ Explains any tradeoffs explicitly
- ✅ Consistent decisions across runs

**Key Issues to Watch:**

The naive prompt creates conflicts:
- "MUST keep diff small" vs "MUST improve readability" 
  → Readable code often means renaming things (larger diff)
  
- "MUST improve performance" vs "MUST maintain compatibility"
  → Better performance might need API changes
  
Without priorities, Claude picks arbitrarily or tries to satisfy all (impossible).

**Key Takeaway:** 
When you have multiple "MUST" constraints that can conflict, Claude needs a tiebreaker. Priority ordering gives Claude clear rules for making tradeoffs.

## Part 3: Non-Code Example - Product Analytics Report

Let's see constraint priority in a real business scenario

In [ ]:
# Product analytics data
analytics_data = """
Q1 Product Metrics:

Feature Adoption:
- New dashboard: 45% of users tried it, 12% use it daily
- AI recommendations: 67% enabled, 8% conversion rate (low)
- Export to PDF: 89% adoption, heavily used
- Mobile app: 23% of users have it installed, 5% DAU

Performance Issues:
- Page load time increased from 1.2s to 3.8s (search page)
- API timeout rate: 0.3% (up from 0.1%)
- Mobile crash rate: 2.1% (industry average: 1%)

User Segments:
- Enterprise (20% of users, 70% of revenue): High satisfaction, want SSO
- SMB (50% of users, 25% of revenue): Complaining about pricing
- Free tier (30% of users, 0% revenue): High churn, support load

Customer Feedback:
- #1 request: Better mobile app (234 votes)
- #2 request: Lower pricing (189 votes)
- #3 request: SSO/SAML (156 votes)
- #4 request: Faster search (98 votes)

Engineering Capacity: 3 developers, 2 are allocated to enterprise contract commitments
"""

print("Your task: Analyze this data and recommend what to build next quarter")
print(analytics_data)

In [ ]:
# Naive: Flat constraints with conflicting priorities
naive_analytics_prompt = f"""Analyze this product data and recommend what to build next quarter:

{analytics_data}

CONSTRAINTS:
- You MUST prioritize features that drive revenue
- You MUST listen to user feedback and votes
- You MUST fix critical performance issues
- You MUST improve key metrics (DAU, retention, conversion)
- You MUST respect engineering capacity constraints
- You MUST focus on data-driven decisions

Provide your recommendations.
"""

response_naive_analytics = client.messages.create(
    model="claude-opus-4-20250514",
    max_tokens=2000,
    temperature=0.2,
    messages=[
        {"role": "user", "content": naive_analytics_prompt}
    ]
)

print("NAIVE (flat constraints):")
print(response_naive_analytics.content[0].text)
print("\n" + "="*80 + "\n")

In [ ]:
# Optimized: Clear priority ordering
optimized_analytics_prompt = f"""<data>
{analytics_data}
</data>

<task>
Analyze this product data and recommend 2-3 priorities for next quarter.
</task>

<constraints>
When making tradeoff decisions, follow this priority order:

Priority 1 (CRITICAL): Protect revenue
- Don't risk losing enterprise customers (70% of revenue)
- Fix issues that directly cause cancellations

Priority 2 (HIGH): Respect engineering capacity
- Only 1 developer available for new work
- Be realistic about what can ship in 90 days

Priority 3 (MEDIUM): Fix user-impacting bugs before features
- Performance issues that hurt existing users take precedence
- But: consider impact vs effort

Priority 4 (LOW): User vote count
- Votes signal interest, but might not align with business goals
- Free-tier users vote but don't pay

Priority 5 (LOWEST): Improve vanity metrics
- Growing DAU/MAU is nice but not the goal
- Revenue and retention matter more

When votes conflict with revenue, choose revenue.
When performance issues conflict with new features, fix performance first UNLESS the feature is critical for enterprise retention.
</constraints>

<output>
Recommend 2-3 initiatives with rationale explaining which constraints drove each decision.
</output>
"""

response_optimized_analytics = client.messages.create(
    model="claude-opus-4-20250514",
    max_tokens=2000,
    temperature=0.2,
    messages=[
        {"role": "user", "content": optimized_analytics_prompt}
    ]
)

print("OPTIMIZED (explicit priority order):")
print(response_optimized_analytics.content[0].text)
print("\n" + "="*80 + "\n")

## Product Analytics: What to Observe

**The Conflicting Signals:**
- Most votes: Mobile app (234) → but only 5% DAU, mostly free users
- Performance issue: Search (3.8s) → affects all users
- Revenue driver: SSO (156 votes) → enterprise wants it (70% of revenue)
- Capacity: Only 1 developer available

**Naive Approach Will Likely:**
- Try to balance everything
- Recommend mobile app (most votes)
- Recommend search fix (clear problem)
- Recommend SSO (revenue)
- Recommend improving AI recommendations (low conversion)
→ **Result: 4-5 initiatives for 1 developer = nothing ships**

**Optimized Approach Should:**
1. **SSO for enterprise** (Priority 1: protect 70% of revenue)
2. **Fix search performance** (Priority 3: but affects all paying users)
3. **Skip mobile app** (Priority 4: votes from non-payers, low actual usage)

**Key Conflicts the Priorities Resolve:**
- "Listen to user votes" vs "Drive revenue" → Revenue wins (mobile app deprioritized)
- "Fix performance" vs "Respect capacity" → Fix the one that matters most (search > mobile crashes)
- "Improve metrics" vs "Protect revenue" → Enterprise needs beat growth metrics

**This is where priority ordering is critical** - the data has conflicting signals, and without priorities Claude will either:
1. Try to do everything (unrealistic)
2. Pick based on its own judgment (inconsistent)
3. Default to "loudest voice" (user votes)

# Exercise 05: Action Policy (Do vs Suggest)

## The Scenario
You have a product analytics dataset and you want Claude to analyze it, identify issues, and recommend fixes.

**The Problem:** Without explicit action policy, Claude often:
- SUGGESTS analysis instead of DOING it
- Asks "would you like me to..." instead of just analyzing
- Stops after one insight when you need comprehensive analysis
- Doesn't follow through on implications

**Goal**: Compare vague requests vs explicit action policy

In [ ]:
# Product data with hidden issues
funnel_data = """
User Onboarding Funnel (Last 30 days):

Step 1 - Signup page visit: 10,000 users
Step 2 - Create account: 7,500 users (75% conversion)
Step 3 - Email verification: 4,200 users (56% conversion) 
Step 4 - Complete profile: 3,800 users (90% conversion)
Step 5 - First action: 950 users (25% conversion)
Step 6 - Second session (7-day): 380 users (40% retention)

Cohort Data:
- Users who completed profile in < 5 min: 1,200 (of 3,800)
  → 45% took first action (540 users)
- Users who took > 5 min: 2,600 (of 3,800)
  → 15% took first action (410 users)

Feature Usage (among activated users):
- Export data: 85% usage
- Invite team: 12% usage
- Connect integration: 8% usage
- Custom dashboard: 3% usage

Time on Platform:
- < 1 minute: 2,100 users (55% of step 4 completers)
- 1-5 minutes: 1,200 users (32%)
- 5-15 minutes: 400 users (10%)
- 15+ minutes: 100 users (3%)
"""

print("DATA: User onboarding funnel")
print(funnel_data)

## Part 1: Naive Approach (Vague request)

In [ ]:
# Naive: Vague request
naive_action_prompt = f"""Analyze this onboarding funnel and identify problems:

{funnel_data}
"""

response_naive_action = client.messages.create(
    model="claude-opus-4-20250514",
    max_tokens=2000,
    temperature=0.2,
    messages=[
        {"role": "user", "content": naive_action_prompt}
    ]
)

print("NAIVE (vague request, no action policy):")
print(response_naive_action.content[0].text)
print("\n" + "="*80 + "\n")

## Part 2: Optimized Approach (Explicit action policy)

In [ ]:
# Optimized: Explicit action policy
optimized_action_prompt = f"""<data>
{funnel_data}
</data>

<task>
Analyze this onboarding funnel comprehensively and provide actionable recommendations.
</task>

<action_policy>
DEFAULT BEHAVIOR:
- Perform COMPLETE analysis, don't stop after finding one issue
- Calculate all relevant conversion rates and metrics
- Identify ALL bottlenecks in the funnel, not just the most obvious one
- For each issue found, provide a specific, testable recommendation
- Prioritize issues by impact (users affected × severity)

DO NOT:
- Ask "would you like me to analyze X?" - just do it
- Stop after identifying one problem - find all major issues
- Give generic advice like "improve onboarding" - be specific
- Skip calculations - show your math
- Suggest solutions without explaining expected impact

REQUIRED OUTPUTS:
1. Conversion rate analysis for each step
2. Identification of the biggest drop-off point(s)
3. Cohort comparison insights
4. Specific, actionable recommendations with expected impact
5. Prioritization of recommendations
</action_policy>
"""

response_optimized_action = client.messages.create(
    model="claude-opus-4-20250514",
    max_tokens=2500,
    temperature=0.2,
    messages=[
        {"role": "user", "content": optimized_action_prompt}
    ]
)

print("OPTIMIZED (explicit action policy):")
print(response_optimized_action.content[0].text)
print("\n" + "="*80 + "\n")

## Comparison: What to Observe

**The Hidden Issues in This Data:**
1. **Email verification drop-off**: 56% conversion (huge problem - losing 3,300 users)
2. **Profile → First action**: 25% conversion (only 950 of 3,800 take action)
3. **Speed matters**: Fast completers (< 5 min) have 3x activation rate (45% vs 15%)
4. **Time on platform**: 55% spend < 1 minute after completing profile (confused/lost)

**Naive Approach Will Likely:**
- Identify 1-2 obvious issues
- Stop after mentioning email verification problem
- Give generic advice ("improve onboarding")
- Miss the speed/time correlation
- Not calculate impact or prioritize

**Optimized Approach Should:**
- Calculate conversion at EVERY step
- Find ALL major bottlenecks (email verification, first action, time on platform)
- Notice the cohort difference (speed matters)
- Provide specific recommendations:
  - "Add email resend button" (not just "fix email verification")
  - "Shorten profile form to < 5 min" (based on cohort data)
  - "Add guided first action" (25% → target 50%)
- Prioritize by impact (email = 3,300 users lost)

**Key Difference:**
- Naive: Passive analysis, stops early, asks permission
- Optimized: Proactive deep dive, complete analysis, actionable output

**This is the "Do vs Suggest" pattern:**
Without action policy, Claude treats analysis as collaborative ("shall I...?")
With action policy, Claude treats it as a job to complete thoroughly

# Exercise 06: Reasoning Control

## The Scenario
You need Claude to answer BOTH simple and complex product analytics questions.

**The Problem:** Using "think step by step" for everything makes:
- Simple questions bloated and slow
- Complex questions might work, but wastes tokens on easy stuff

**Goal**: Control HOW MUCH reasoning Claude shows based on question complexity

In [ ]:
# Two different questions - one simple, one complex

simple_question = """
What's the formula for calculating customer lifetime value (CLV)?
"""

complex_question = """
Our retention rate dropped from 85% to 78% over the last quarter.

Data points:
- Cohort A (Jan signups): 80% 30-day retention → now 72% 90-day retention
- Cohort B (Feb signups): 82% 30-day retention → now 75% 90-day retention  
- Cohort C (Mar signups): 85% 30-day retention (too early for 90-day)
- Feature usage: Power users (10+ sessions/month) retention stayed at 92%
- Support tickets: Up 23% (mostly "can't find X" and "how do I Y")
- New feature launch: Advanced filters launched mid-February
- Competitor: Major competitor dropped prices 20% in March

What's causing the retention drop and what should we do?
"""

print("SIMPLE QUESTION:")
print(simple_question)
print("\nCOMPLEX QUESTION:")
print(complex_question)

## Part 1: Generic "Think Step by Step" (Applied to Both)

In [ ]:
# Generic CoT system prompt (used for both questions)
generic_cot_system = """You are a product analytics expert. 
Always think step by step and show your reasoning for every question."""

# Test on simple question
response_simple_cot = client.messages.create(
    model="claude-opus-4-20250514",
    max_tokens=1500,
    temperature=0.2,
    system=generic_cot_system,
    messages=[
        {"role": "user", "content": simple_question}
    ]
)

print("SIMPLE QUESTION with 'think step by step':")
print(response_simple_cot.content[0].text)
print(f"\nToken count: ~{len(response_simple_cot.content[0].text.split())} words")
print("\n" + "="*80 + "\n")

# Test on complex question
response_complex_cot = client.messages.create(
    model="claude-opus-4-20250514",
    max_tokens=2000,
    temperature=0.2,
    system=generic_cot_system,
    messages=[
        {"role": "user", "content": complex_question}
    ]
)

print("COMPLEX QUESTION with 'think step by step':")
print(response_complex_cot.content[0].text)
print(f"\nToken count: ~{len(response_complex_cot.content[0].text.split())} words")
print("\n" + "="*80 + "\n")

## Part 2: Task-Scoped Reasoning (Adaptive)

In [ ]:
# Adaptive reasoning system prompt
adaptive_system = """You are a product analytics expert.

<reasoning_style>
Adapt your reasoning depth to the question complexity:

SIMPLE questions (definitions, formulas, single-step calculations):
→ Answer directly in 1-3 sentences
→ No preamble, no "let me think about this"
→ Just provide the answer
→ Examples: "What's the formula for X?", "How do you calculate Y?"

COMPLEX questions (multi-factor analysis, trade-offs, root cause investigation):
→ Use structured analysis:
  1. Restate the problem (1 sentence)
  2. List key observations from the data
  3. Evaluate possible explanations
  4. Recommend specific actions with expected impact
→ Examples: retention drops, funnel issues, A/B test interpretation

When uncertain, err toward concise.
</reasoning_style>
"""

# Test on simple question
response_simple_adaptive = client.messages.create(
    model="claude-opus-4-20250514",
    max_tokens=1500,
    temperature=0.2,
    system=adaptive_system,
    messages=[
        {"role": "user", "content": simple_question}
    ]
)

print("SIMPLE QUESTION with adaptive reasoning:")
print(response_simple_adaptive.content[0].text)
print(f"\nToken count: ~{len(response_simple_adaptive.content[0].text.split())} words")
print("\n" + "="*80 + "\n")

# Test on complex question
response_complex_adaptive = client.messages.create(
    model="claude-opus-4-20250514",
    max_tokens=2000,
    temperature=0.2,
    system=adaptive_system,
    messages=[
        {"role": "user", "content": complex_question}
    ]
)

print("COMPLEX QUESTION with adaptive reasoning:")
print(response_complex_adaptive.content[0].text)
print(f"\nToken count: ~{len(response_complex_adaptive.content[0].text.split())} words")
print("\n" + "="*80 + "\n")

## Comparison: What to Observe

**Simple Question: "What's the CLV formula?"**

**With "think step by step":**
- Expected: 150-300 words
- Bloated answer like: "Let me think about customer lifetime value step by step. First, we need to understand what CLV means. Customer lifetime value is... There are several approaches... Let me break this down..."
- Result: "Just give me the formula!" frustration

**With adaptive reasoning:**
- Expected: 20-50 words
- Direct answer: "CLV = (Average Purchase Value × Purchase Frequency × Customer Lifespan) or CLV = (Average Revenue Per User × Gross Margin) / Churn Rate"
- Result: Fast, useful answer

---

**Complex Question: "Why did retention drop?"**

**With "think step by step":**
- May or may not structure well
- Might still be verbose but unorganized

**With adaptive reasoning:**
- Structured analysis:
  1. **Problem**: 85% → 78% retention drop
  2. **Key observations**: Power users fine (92%), newer cohorts better, support up 23%
  3. **Likely causes**: Advanced filters launched mid-Feb confused casual users
  4. **Recommendations**: Onboarding for new feature, simplify UI, A/B test removal

---

**Key Difference:**
- **Generic CoT**: Same verbose approach for everything
- **Adaptive**: Matches reasoning depth to question complexity

**Why This Matters:**
- **Speed**: Simple questions answered instantly
- **Cost**: Fewer tokens on trivial questions
- **Quality**: Complex questions get proper structure, simple ones don't get bloated

# Exercise 07: Output Schema Enforcement

## The Scenario
You're building an automated analytics pipeline that processes Claude's analysis programmatically.

**You need:**
1. Valid JSON (parseable by `json.loads()`)
2. Matches exact schema
3. NO extra text around it
4. Consistent across runs

**Goal**: Compare loose "return as JSON" vs strict schema enforcement

In [ ]:
# Raw analytics data to analyze
raw_metrics = """
Weekly Product Snapshot:

Users: 12,450 MAU (up 8% from last week)
Revenue: $48,200 MRR (up 12%)
Churn: 15 customers churned (2.1% monthly churn)
NPS: 42 (based on 230 responses)
Top feature requests: Dark mode (156 votes), API access (89 votes), Slack integration (67 votes)
Critical bug reported: Export function failing for files >10MB (affects ~5% of exports)
Support ticket volume: 234 tickets (up 45% - mostly about new pricing page confusion)
A/B test running: New onboarding flow - 12% better activation but 8% more support tickets
Competitor news: CompetitorX raised $20M Series B, aggressively targeting our customer segment
"""

print("RAW METRICS TO ANALYZE:")
print(raw_metrics)

## Part 1: Naive Approach (Loose "return as JSON")

In [ ]:
# Naive: Vague JSON request
naive_schema_prompt = f"""Analyze this product metrics snapshot and return as JSON with:
- Overall health status
- Top 3 priorities
- Risk level
- Recommended actions

Data:
{raw_metrics}
"""

response_naive_schema = client.messages.create(
    model="claude-opus-4-20250514",
    max_tokens=1500,
    temperature=0.2,
    messages=[
        {"role": "user", "content": naive_schema_prompt}
    ]
)

result_naive_schema = response_naive_schema.content[0].text
print("NAIVE (loose JSON request):")
print(result_naive_schema)
print("\n" + "="*80 + "\n")

# Try to parse
try:
    parsed = json.loads(result_naive_schema)
    print("✅ JSON is valid")
    print(f"Keys: {list(parsed.keys())}")
except json.JSONDecodeError as e:
    print(f"❌ JSON parsing FAILED: {e}")
    print("\nAttempting to extract JSON...")
    # Try to find JSON between first { and last }
    if '{' in result_naive_schema and '}' in result_naive_schema:
        start = result_naive_schema.find('{')
        end = result_naive_schema.rfind('}') + 1
        json_only = result_naive_schema[start:end]
        try:
            parsed = json.loads(json_only)
            print("✅ JSON found after cleanup")
            print(f"Keys: {list(parsed.keys())}")
        except:
            print("❌ Still can't parse")

## Part 2: Optimized Approach (Strict Schema Enforcement)

In [ ]:
# Optimized: Strict schema enforcement
optimized_schema_prompt = f"""<data>
{raw_metrics}
</data>

<task>
Analyze these product metrics and provide structured insights.
</task>

<output_format>
Return ONLY valid JSON matching this EXACT schema:

{{
  "health_status": "healthy" | "concern" | "critical",
  "risk_level": 1-10,
  "top_priorities": [
    {{
      "priority": "string",
      "impact": "high" | "medium" | "low",
      "urgency": "immediate" | "this_week" | "this_month"
    }}
  ],
  "metrics_summary": {{
    "growth_trend": "positive" | "neutral" | "negative",
    "churn_status": "normal" | "elevated" | "critical",
    "revenue_trend": "up" | "flat" | "down"
  }},
  "immediate_actions": ["string"]
}}

CRITICAL RULES:
1. Response must start with {{ and end with }}
2. NO markdown code blocks (no ```)
3. NO explanatory text before or after the JSON
4. NO extra fields beyond this schema
5. Use null for values that cannot be determined
6. top_priorities must have exactly 3 items
7. immediate_actions must have 2-4 items
</output_format>
"""

response_optimized_schema = client.messages.create(
    model="claude-opus-4-20250514",
    max_tokens=1500,
    temperature=0.2,
    messages=[
        {"role": "user", "content": optimized_schema_prompt}
    ]
)

result_optimized_schema = response_optimized_schema.content[0].text
print("OPTIMIZED (strict schema enforcement):")
print(result_optimized_schema)
print("\n" + "="*80 + "\n")

# Try to parse
try:
    parsed_optimized = json.loads(result_optimized_schema)
    print("✅ JSON is valid")
    print(f"\nSchema validation:")
    print(f"  health_status: {parsed_optimized.get('health_status')}")
    print(f"  risk_level: {parsed_optimized.get('risk_level')}")
    print(f"  top_priorities count: {len(parsed_optimized.get('top_priorities', []))}")
    print(f"  immediate_actions count: {len(parsed_optimized.get('immediate_actions', []))}")
    print(f"  All expected keys present: {set(parsed_optimized.keys()) == {'health_status', 'risk_level', 'top_priorities', 'metrics_summary', 'immediate_actions'}}")
    
    # Pretty print for readability
    print("\nParsed structure:")
    print(json.dumps(parsed_optimized, indent=2))
except json.JSONDecodeError as e:
    print(f"❌ JSON parsing FAILED: {e}")

## Comparison: What to Observe

**Common Problems with Naive Approach:**

1. **Markdown wrapper**:
   ```json
   {
     "health": "good"
   }
   ```
   → Breaks `json.loads()`

2. **Explanatory text**:
   "Here's my analysis:\n{...}\nLet me know if you need more details."
   → Need to extract JSON manually

3. **Schema drift**:
   - Uses "health" instead of "health_status"
   - Adds extra fields like "notes" or "confidence"
   - Wrong types: `"risk_level": "high"` instead of number

4. **Inconsistent structure**:
   - Sometimes 2 priorities, sometimes 5
   - Field names change between runs

---

**Optimized Approach Benefits:**

✅ **Direct parsing**: `json.loads()` works immediately, no cleanup needed

✅ **Exact schema**: Always has the same fields with correct types

✅ **Predictable**: Can write code that depends on structure
```python
if parsed['risk_level'] > 7:
    alert_team()
```

✅ **Pipeline ready**: Can feed directly into dashboards, databases, etc.

---

**Why This Matters for Product Analytics:**

You're building automated systems that:
- Pull Claude's analysis into dashboards
- Trigger alerts based on risk_level
- Store insights in databases
- Feed into other systems

**Without strict schema**: You need error handling, cleanup code, and manual QA

**With strict schema**: Direct integration, reliable automation

**Real Cost**: 
- Naive = 30 min building JSON extraction logic + ongoing maintenance
- Optimized = 5 min writing schema, works forever

# Exercise 08: Full Integration

## The Challenge
Combine ALL 7 techniques into a single, production-quality prompt for product analytics.

**Scenario**: Build a "Product Health Analyzer" that:
1. Takes raw product metrics
2. Analyzes for risks and opportunities
3. Returns structured JSON with actionable insights
4. Provides priority levels and recommendations

**Goal**: Use all 7 techniques together in one integrated prompt

In [ ]:
# Complex product metrics dataset
complex_metrics = """
Product Metrics - Weekly Snapshot:

USER METRICS:
- MAU: 45,200 (down 3% WoW - first decline in 8 weeks)
- WAU: 18,500 (down 2% WoW)
- DAU: 6,800 (down 5% WoW)
- New signups: 1,850 (up 12% WoW)
- Activation rate: 23% (down from 31% last week)

ENGAGEMENT:
- Avg session duration: 8.2 min (down from 11.5 min)
- Sessions per user: 3.1 (down from 4.2)
- Feature A usage: 78% of MAU (stable)
- Feature B usage: 12% of MAU (down from 18% - launched 3 weeks ago)
- Feature C usage: 4% of MAU (beta, expected)

REVENUE:
- MRR: $182,400 (up 8% WoW)
- New MRR: $18,200
- Churn MRR: $6,500
- Expansion MRR: $12,100
- ARPU: $4.03 (up from $3.87)

CUSTOMER HEALTH:
- Churned users: 38 (up 90% - highest in 6 months)
- At-risk users (no activity 7+ days): 2,100 (up 45%)
- NPS: 38 (down from 44)
- Support tickets: 456 (up 67% - mostly "where did feature B go?")
- Feature requests mentioning "confused": 89 instances

PRODUCT CHANGES:
- Week -3: Launched Feature B (new workflow automation)
- Week -2: Moved Feature B to new navigation menu
- Week -1: Updated pricing page (no price changes)
- This week: Released bug fix for export function

EXTERNAL:
- Competitor X launched free tier (our free tier is 14-day trial)
- Industry conference happening (we're not attending)
- Economic news: Tech layoffs announced at major companies
"""

print("COMPLEX PRODUCT METRICS:")
print(complex_metrics)

## The Full Integration Prompt

This prompt combines ALL 7 techniques:
1. ✅ XML Structure
2. ✅ System vs User separation
3. ✅ Explicit intent (WHY)
4. ✅ Constraint priority
5. ✅ Action policy
6. ✅ Reasoning control
7. ✅ Schema enforcement

In [ ]:
# INTEGRATED PROMPT - All 7 techniques combined

# === SYSTEM PROMPT === (Technique #2: Role Separation)
integrated_system = """You are a Senior Product Analytics Lead with 10+ years experience.

Your core values:
- Truth over optimism: Call out real problems even if uncomfortable
- Impact over volume: Focus on what matters most
- Action over analysis: Provide specific, testable recommendations

<action_policy>
DEFAULT BEHAVIOR:
- Perform COMPLETE analysis across all metric categories
- Calculate all relevant rates and correlations
- Identify ALL significant issues, not just the most obvious
- For each issue, provide specific recommendations with expected impact
- Prioritize by: (users affected) × (severity) × (urgency)

DO NOT:
- Stop after finding one problem
- Give generic advice like "improve engagement"
- Ignore correlations between metrics
- Skip calculations - show your math
</action_policy>

<reasoning_style>
For product health analysis (complex, multi-factor):
- List key observations organized by category
- Identify root causes by correlating metrics
- Evaluate urgency and impact
- Provide specific, measurable recommendations

Keep descriptions concise - one line per observation.
</reasoning_style>
"""

# === USER MESSAGE === (Techniques #1, 3, 4, 5, 6, 7)
integrated_user = f"""<context>
{complex_metrics}
</context>

<instructions>
1. Analyze all metric categories for issues and opportunities
2. Correlate metrics to identify root causes (not just symptoms)
3. Assess severity and urgency for each issue
4. Provide specific, testable recommendations
</instructions>

<why_this_matters>
This analysis feeds our weekly exec dashboard and triggers automated alerts.
- Critical issues trigger immediate team mobilization
- High priority issues go into sprint planning
- The team acts on these insights - vague advice wastes time
- Support tickets surged 67% - users are struggling RIGHT NOW
</why_this_matters>

<constraints>
When making tradeoff decisions, follow this priority:

Priority 1 (CRITICAL): Identify threats to revenue
- Churn spikes, expansion drops, at-risk users
- These directly impact company survival

Priority 2 (HIGH): Find engagement problems causing churn
- Usage drops, session declines, feature confusion
- Leading indicators of future churn

Priority 3 (MEDIUM): Spot growth opportunities
- Expansion potential, feature adoption gaps
- Important but not urgent

Priority 4 (LOW): Note external factors
- Competitor moves, market conditions
- Context, not action items

When engagement drops AND churn rises, prioritize the engagement issue (it's the root cause).
</constraints>

<output_format>
Return ONLY valid JSON matching this EXACT schema:

{{
  "health_summary": {{
    "status": "healthy" | "concerning" | "critical",
    "risk_score": 1-10,
    "primary_concern": "string (one line)"
  }},
  "critical_issues": [
    {{
      "issue": "string (max 60 chars)",
      "severity": "critical" | "high" | "medium",
      "users_affected": number,
      "root_cause": "string",
      "recommendation": "string (specific and testable)",
      "expected_impact": "string (quantified if possible)"
    }}
  ],
  "key_correlations": [
    {{
      "metrics": ["string", "string"],
      "insight": "string"
    }}
  ],
  "immediate_actions": [
    "string (specific action with owner/timeline)"
  ]
}}

CRITICAL RULES:
1. Response must start with {{ and end with }}
2. NO markdown code blocks (no ```)
3. NO explanatory text before or after JSON
4. NO extra fields beyond this schema
5. critical_issues must have 2-4 items (ordered by priority)
6. key_correlations must have 2-3 items
7. immediate_actions must have 2-3 items
8. Use exact field names (case-sensitive)
9. severity must be lowercase
10. All numbers must be actual numbers, not strings
</output_format>
"""

print("System Prompt:")
print(integrated_system)
print("\n" + "="*80 + "\n")
print("User Message:")
print(integrated_user[:500] + "...[truncated]")

In [ ]:
# Run the integrated prompt
response_integrated = client.messages.create(
    model="claude-opus-4-20250514",
    max_tokens=3000,
    temperature=0.2,
    system=integrated_system,
    messages=[
        {"role": "user", "content": integrated_user}
    ]
)

result_integrated = response_integrated.content[0].text
print("INTEGRATED PROMPT RESULT:")
print(result_integrated)
print("\n" + "="*80 + "\n")

# Validate the output
try:
    parsed_integrated = json.loads(result_integrated)
    print("✅ JSON is valid!")
    
    # Check schema compliance
    expected_keys = {"health_summary", "critical_issues", "key_correlations", "immediate_actions"}
    actual_keys = set(parsed_integrated.keys())
    
    print(f"\n📊 Schema Validation:")
    print(f"  Expected keys: {expected_keys}")
    print(f"  Actual keys: {actual_keys}")
    print(f"  Match: {expected_keys == actual_keys}")
    
    print(f"\n📈 Content Check:")
    print(f"  Health status: {parsed_integrated['health_summary']['status']}")
    print(f"  Risk score: {parsed_integrated['health_summary']['risk_score']}/10")
    print(f"  Critical issues found: {len(parsed_integrated['critical_issues'])}")
    print(f"  Key correlations found: {len(parsed_integrated['key_correlations'])}")
    print(f"  Immediate actions: {len(parsed_integrated['immediate_actions'])}")
    
    # Pretty print
    print(f"\n📋 Full Analysis:")
    print(json.dumps(parsed_integrated, indent=2))
    
except json.JSONDecodeError as e:
    print(f"❌ JSON parsing FAILED: {e}")

## How the 7 Techniques Work Together

**1. XML Structure (Exercise 01)**
- `<context>`, `<instructions>`, `<constraints>`, `<output_format>` sections
- Clear separation of concerns
- Claude knows what each part is for

**2. System vs User Separation (Exercise 02)**
- System: Role identity, values, persistent behaviors
- User: Task-specific data, instructions, constraints
- Prevents mixing ephemeral and persistent context

**3. Explicit Intent / WHY (Exercise 03)**
- `<why_this_matters>` explains business impact
- "Support tickets surged 67% - users struggling RIGHT NOW"
- Claude understands urgency and prioritizes accordingly

**4. Constraint Priority (Exercise 04)**
- Numbered priorities: Revenue threats > Engagement issues > Growth
- Conflict resolution: "When engagement drops AND churn rises, prioritize engagement"
- Clear tradeoff rules

**5. Action Policy (Exercise 05)**
- "Perform COMPLETE analysis" - don't stop after one issue
- "DO NOT give generic advice" - be specific
- "Show your math" - calculate, don't eyeball

**6. Reasoning Control (Exercise 06)**
- Structured for complex analysis (this IS complex)
- "Keep descriptions concise - one line per observation"
- Not verbose, but thorough

**7. Schema Enforcement (Exercise 07)**
- Exact JSON schema with types
- 10 critical rules for formatting
- "NO markdown, NO extra text, start with {, end with }"

**Result**: Production-ready analysis that:
- ✅ Finds all critical issues (churn spike, feature confusion, engagement drop)
- ✅ Returns perfect JSON every time
- ✅ Provides actionable, specific recommendations
- ✅ Ready to feed into dashboards and alerting systems